# Lab 46: Scaling the signals across workers and traffic

The three [Lab 44](../44-hardening-the-signals/) signals assumed a single process and a curated sample. Scale them: a shared StateStore with an atomic claim (one worker pages), a larger clean held-out reference distilled from real traffic, and a per-document corpus map so a change flags only the affected canaries. Fill in the `TODO` cells; reference in `solution/`.

## Step 0: Setup

In [ ]:
import json
import pathlib
import sys
# Lab 46 scales the three Lab 44 signals across workers and real traffic. Scripts live in
# the operating-the-loop toolkit.
loop = pathlib.Path.cwd().parent / "41-operating-the-loop"
sys.path.insert(0, str(loop))
print("scaling signals in:", loop.name)

## Step 1: A shared store with an atomic claim (item 1)

The check-then-record race, and the fix.

In [ ]:
from store import FileLockStore, InMemoryStore, naive_claim, _race
import os
import tempfile
# TODO: (a) use _race + naive_claim on a fresh file and show BOTH workers win (the bug);
# (b) use _race + a FileLockStore.try_claim and show exactly ONE worker wins (the fix).
raise NotImplementedError

### `deliver` over a shared store

In [ ]:
from notify import deliver, format_alert, to_slack
# deliver now takes a StateStore (shared) OR a dict (legacy single-process). Same call site.
p = format_alert("judged_faithfulness", 0.55, 0.764)
shared = InMemoryStore()
r1, shared = deliver(to_slack(p), p, url=None, store=shared, now=0.0, sleep=lambda s: None)
r2, shared = deliver(to_slack(p), p, url=None, store=shared, now=600.0, sleep=lambda s: None)
print("first worker: ", r1)
print("second worker (same incident, shared store):", r2)

## Step 2: A larger reference from real traffic (item 2)

In [ ]:
from build_reference import clean_candidates, stratify, band_ci
# Item 2: Lab 44's reference was 16 curated queries. A 16-query band has a wide confidence
# interval. Distill a larger clean, stratified, HELD-OUT reference from real captured traffic.
with open(loop / "captured_traffic.jsonl") as f:
    captured = [json.loads(line) for line in f]
with open(loop.parent / "36-training-the-router" / "router_trainset.jsonl") as f:
    trainset = {json.loads(line)["query"] for line in f}
cand = clean_candidates(captured, trainset)
ref = stratify(cand, per_route=12)
print(f"captured {len(captured)} -> cleaned {len(cand)} (dropped garbage/dupes/trainset-leak) -> reference {len(ref)}")

# why bigger: the band's confidence interval narrows as n grows (SE = std / sqrt(n))
spread = [0.80, 0.78, 0.83, 0.76]
print("band CI @ n=16:", band_ci(spread * 4))
print("band CI @ n=80:", band_ci(spread * 20), " <- tighter, so the drift check is less jumpy")

## Step 3: A per-document corpus map (item 3)

In [ ]:
from canary import load_canaries, review_status_per_doc
# TODO: build an old per-doc map and a new one that changes a single doc; call
# review_status_per_doc and confirm only canaries whose corpus_refs include that doc are
# flagged - fewer than the whole-corpus review (all corpus-dependent canaries).
raise NotImplementedError

## Step 4: The scaled cadence

In [ ]:
# The scaled cadence (workflows updated in place):
#   rag-faithfulness-nightly  notify uses --shared-store (atomic across workers)
#   rag-drift-check           caches the per-document corpus map so review can diff runs
#   rag-maintenance-loop      promote refreshes the reference from traffic, then re-baselines
print("Alerts dedup across workers, the baseline tracks real traffic, and corpus review")
print("only wakes you for the canaries a change actually touched.")

## Step 5: The theme

In [ ]:
# The theme: anything you kept as a single point breaks when production makes it plural.
#  - single-process state -> a shared store with an atomic claim (one worker pages);
#  - a curated 16-query reference -> a sampled, verified, stratified set with a tighter band;
#  - one whole-corpus hash -> a per-document map that localizes change.
print("Distribute the state, sample the reference, and localize the change - or the signal")
print("that worked on one box lies the moment you run two.")

## What you built

The across-workers, real-traffic version of the three signals: `store.py` provides a `StateStore` with an atomic `try_claim` (in-memory for one process; `FileLockStore` for a shared filesystem, standing in for Redis/DB), and `notify.deliver` now takes a store so exactly one worker pages for an incident; `build_reference.py` distills a larger clean, stratified, held-out reference from captured traffic, with a confidence interval that narrows as the sample grows; and `canary.py` keeps a per-document fingerprint map so a corpus change flags only the canaries that depend on the changed docs.

**Where this simplifies:** `FileLockStore` needs a shared filesystem and uses an OS file lock - a real fleet uses Redis (`SETNX`/`SET NX PX`) or a DB row lock; the claim is taken before delivery, so a send that fails after retries still consumed the cooldown slot (release-on-failure is a refinement); the reference distiller's clean filter is heuristic (length, dedup, trainset-leak) and still needs human or model verification; and the per-document map keys on file content, so a no-op reformat still counts as a change.

Next: [Lab 47](../47-trustworthy-gold/) makes the evaluation anchor itself plural - gold from multiple experts, with the judge ceiling re-derived against it.